# DermArbiter text-only harness benchmark on a free GPU (Kaggle)

Zero API cost: an open-weight model is served by **Ollama** on the notebook GPU (the same
server and JSON-schema behaviour as the local pilot), and the benchmark sends cases
concurrently. Settings: **Accelerator = GPU T4 x2**, **Internet = on**.

1. Edit the settings cell and run all cells.
2. Output goes to `/kaggle/working/runs` and `runs.zip`. Save a version (or download the zip)
   before the session ends.
3. A stopped run resumes when you rerun it. Delete the unfinished arm folder first; the
   driver names it.

Test-split runs need frozen data. See the last cell.

In [ ]:
# ---- settings ----
MODEL = "qwen2.5:7b"          # second backbone, e.g. "gemma3:12b"
TRACKS = ["scin_text_k30", "medmcqa_skin"]   # + "medxpertqa_derm" (test only)
SPLIT = "dev"
LIMITS = {"scin_text_k30": 20, "medmcqa_skin": 20, "medxpertqa_derm": 48}
# main run: SPLIT = "test", LIMITS = {"scin_text_k30": 600, "medmcqa_skin": 400, "medxpertqa_derm": 48}
WORKERS = 8                   # concurrent cases; must not exceed OLLAMA_NUM_PARALLEL
BRANCH = "research/text-only-harness"

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("DermArbiter"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/Furkanahii/DermArbiter"], check=True)
os.chdir("/kaggle/working/DermArbiter")
!git log --oneline -1

## Data: rebuild from pinned public sources and check hashes against the tracked reports

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/raw && cd /kaggle/working/raw
[ -f scin_cases.csv ] || curl -sSO https://storage.googleapis.com/dx-scin-public-data/dataset/scin_cases.csv
[ -f scin_labels.csv ] || curl -sSO https://storage.googleapis.com/dx-scin-public-data/dataset/scin_labels.csv
[ -f medmcqa_train.parquet ] || curl -sSL -o medmcqa_train.parquet https://huggingface.co/datasets/openlifescienceai/medmcqa/resolve/91c6572c454088bf71b679ad90aa8dffcd0d5868/data/train-00000-of-00001.parquet
[ -f medxpertqa_text_test.jsonl ] || curl -sSL -o medxpertqa_text_test.jsonl https://huggingface.co/datasets/TsinghuaC3I/MedXpertQA/resolve/7e7c465a68eb2b866926bfa59c8c9d17a8daba65/Text/test.jsonl
cd /kaggle/working/DermArbiter
mkdir -p /kaggle/working/built
python scripts/build_text_benchmark.py scin --cases ../raw/scin_cases.csv --labels ../raw/scin_labels.csv --output /kaggle/working/built/scin_text_k30.jsonl > /dev/null
python scripts/build_text_benchmark.py medmcqa --parquet ../raw/medmcqa_train.parquet --revision 91c6572c454088bf71b679ad90aa8dffcd0d5868 --output /kaggle/working/built/medmcqa_skin.jsonl > /dev/null
python scripts/build_text_benchmark.py medxpertqa --jsonl ../raw/medxpertqa_text_test.jsonl --revision 7e7c465a68eb2b866926bfa59c8c9d17a8daba65 --output /kaggle/working/built/medxpertqa_derm.jsonl > /dev/null

In [ ]:
import hashlib, json
for t in ["scin_text_k30", "medmcqa_skin", "medxpertqa_derm"]:
    got = hashlib.sha256(open(f"/kaggle/working/built/{t}.jsonl", "rb").read()).hexdigest()
    want = json.load(open(f"data/text_benchmark/{t}.report.json"))["output_sha256"]
    print(t, "OK" if got == want else f"MISMATCH {got} != {want}")
    assert got == want, "Built data differs from the tracked build: stop and investigate"

## Model server (Ollama on the notebook GPUs)

In [ ]:
%%bash
command -v ollama >/dev/null || (curl -fsSL https://ollama.com/install.sh | sh) > /dev/null
pgrep -x ollama >/dev/null || (OLLAMA_NUM_PARALLEL=8 OLLAMA_MAX_LOADED_MODELS=1 nohup ollama serve > /kaggle/working/ollama.log 2>&1 &)
for i in $(seq 60); do curl -s localhost:11434/api/version >/dev/null && break; sleep 2; done
curl -s localhost:11434/api/version

In [ ]:
!ollama pull {MODEL} 2>&1 | tail -1
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

Check the returned model string and that the reply follows the JSON schema (`differential` from the candidate list, numeric `confidence`, short `rationale`).

In [ ]:
!python scripts/run_text_benchmark.py probe /kaggle/working/built/scin_text_k30.jsonl --split dev \
    --provider openai_compatible --base-url http://127.0.0.1:11434/v1 --model {MODEL} --timeout 600

## Run all ten arms (resumable) and analyse

In [ ]:
import shlex
flags = "--dev-pilot" if SPLIT == "dev" else ""
for t in TRACKS:
    if t == "medxpertqa_derm" and SPLIT == "dev":
        continue                                   # MedXpertQA has a test split only
    cmd = (f"python scripts/run_pilot.py /kaggle/working/built/{t}.jsonl --split {SPLIT} --limit {LIMITS[t]} {flags} "
           f"--provider openai_compatible --base-url http://127.0.0.1:11434/v1 "
           f"--model {MODEL} --expected-model-version {MODEL} --workers {WORKERS} --timeout 900 "
           f"--out /kaggle/working/runs/{t}_{SPLIT}{LIMITS[t]}")
    print("$", cmd, flush=True)
    subprocess.run(shlex.split(cmd), check=True)

In [ ]:
import glob
for t in TRACKS:
    out = f"/kaggle/working/runs/{t}_{SPLIT}{LIMITS[t]}"
    runs = sorted(glob.glob(f"{out}/*/*_r0"))
    if runs:
        subprocess.run(["python", "scripts/analyze_runs.py", f"/kaggle/working/built/{t}.jsonl", "--split", SPLIT,
                        "--limit", str(LIMITS[t]), "--runs", *runs, "--out", f"{out}/analysis"], check=True)
!cd /kaggle/working && zip -qr runs.zip runs ollama.log && ls -lh runs.zip

## Test split (only after the data review)

Live runs refuse non-frozen data. After the team has reviewed the build
(`docs/data_review_tr.md`), rebuild each track with
`--data-status frozen --accepted-by "<reviewer name>"`. Frozen files have different
hashes, so skip the hash-check cell. Then set `SPLIT = "test"`
and rerun the run cells (`LIMITS` above). Do not change prompts or settings after looking at test results.